# Entendimento dos dados e EDA

## 1. Objetivo

Este notebook tem como objetivo explorar e compreender os dados do dataset Olist, identificando padrões, inconsistências e oportunidades de análise que suportem a modelagem preditiva e a tomada de decisão.

A análise inclui:
- Estrutura dos dados
- Qualidade dos dados
- Distribuições
- Relações entre variáveis
- Padrões temporais

## 2. Importação das Bibliotecas 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import sqlite3

from IPython.display import Markdown
from IPython.core.display import HTML
import math

import glob
import warnings
warnings.filterwarnings('ignore')

## 3. Configurações Iniciais


In [2]:
# Cor principal do projeto
PRIMARY_COLOR = "#50e550"
SECONDARY_COLORS = sns.light_palette(PRIMARY_COLOR, n_colors=5)

# Estilo geral
sns.set_theme(style="whitegrid")

# Tamanho padrão
plt.rcParams['figure.figsize'] = (10, 6)

# Fonte
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

## 4. Conexão com banco de dados SQLite 

In [3]:
# conectar ao banco
conn = sqlite3.connect('../data/raw/olist.sqlite')

# listar tabelas
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, conn)

tables

,name
0,product_category_name_translation
1,sellers
2,customers
3,geolocation
4,order_items
5,order_payments
6,order_reviews
7,orders
8,products
9,leads_qualified


O banco de dados possui multiplas tabelas 

## 5. Exploração Inicial das Tabelas

### 5.1 Orders 

In [4]:
orders = pd.read_sql("SELECT * FROM orders", conn)
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [6]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [7]:
orders.describe(include='all')

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


In [8]:
orders['order_status'].unique()

<ArrowStringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

#### 👉 Observar:

order_id → identificador do pedido  

customer_id → cliente  

order_status → status do pedido  

order_purchase_timestamp → data da compra  

order_approved_at → aprovação do pagamento  

order_delivered_carrier_date → envio ao transportador  

order_delivered_customer_date → entrega final  

order_estimated_delivery_date → previsão de entrega  

- datas estão com o tipo de string será preciso transformar em datetime 
- Status do pedido 
- datas de entrega 
- data estimada pela transportadora

A tabela orders é fundamental para o projeto, pois contém as informações temporais necessárias para a construção da série histórica de vendas.

Além disso, as datas de entrega permitem estimar o lead time logístico, variável essencial para o cálculo de estoque ideal e ponto de reposição.

A variável order_status será utilizada para filtrar apenas pedidos concluídos, garantindo maior confiabilidade na análise de demanda.


### 5.2 Order_items

In [9]:
order_items = pd.read_sql("SELECT * FROM order_items LIMIT 5", conn)
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [10]:
order_items.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')

In [11]:
order_items.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

In [12]:
order_items.describe()

,order_item_id,price,freight_value
count,5.0,5.000000,5.000000
mean,1.0,142.138000,16.404000
std,0.0,99.668093,3.176221
min,1.0,12.990000,12.790000
25%,1.0,58.900000,13.290000
50%,1.0,199.000000,17.870000
75%,1.0,199.900000,18.140000
max,1.0,239.900000,19.930000


#### 👉 Observar:

order_id → pedido  
order_item_id → item dentro do pedido  
product_id → produto  
seller_id → vendedor (fornecedor)  
shipping_limit_date → prazo de envio  
price → preço do produto  
freight_value → custo de frete  


#### Variavel product_id 

podemos usar para:
- prever demanda por produto
- analisar consumo
- calcular estoque ideal

### Variavel Price

base para:
- receita
- custo de compra(simulado)
- análise financeira

#### variavel freight_value

podemos:

- calcular o custo real = preço + frete

#### variavel seller_id
Pode ser usado como:

✔️ fornecedor - possui prazos de entrega confiaveis os melhores preços
✔️ comparação de custo 
✔️ simulação de supply chain

💡 Insight avançado:

> Você pode descobrir qual “fornecedor” é mais eficiente (menor custo total)

### variavel order_item_id

quantidade de itens no mesmo pedido

#### variavel shipping_limit_date

Prazo máximo de envio

Pode ajudar a:

analisar atraso logístico
entender SLA

### 5.3 products

In [13]:
products = pd.read_sql("SELECT * FROM products", conn)
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [14]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [15]:
products.dtypes

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [16]:
products.describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


#### 👉 Observar:
product_id → identificador  
product_category_name → categoria  
product_name_lenght → tamanho do nome  
product_description_lenght → tamanho da descrição  
product_photos_qty → nº de fotos  

product_weight_g → peso  
product_length_cm → comprimento  
product_height_cm → altura  
product_width_cm → largura 

#### atraves desta tabela podemos criar a Feature Volume 

Isso permite:
- analisar impactos logísticos
- explicar variação de frete
- melhorar decisões de estoque e compra

🔥 Insight 

👉 Produtos maiores/pesados:

maior custo de transporte
maior custo de armazenagem
maior impacto financeiro

👉 Ou seja:

> 💡 não é só quanto vender — é quanto custa manter


**A tabela products fornece informações relevantes sobre as características físicas e categóricas dos produtos.**

As variáveis de peso e dimensões permitem estimar o impacto logístico, sendo fundamentais para análises de custo e consumo inteligente.

A categorização dos produtos possibilita análises segmentadas, contribuindo para a identificação de padrões de demanda e otimização de decisões de estoque.

### 5.4 customers

In [17]:
customers = pd.read_sql("SELECT * FROM customers", conn)
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [18]:
customers.dtypes

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

In [19]:
customers.describe()

,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000


In [20]:
customers.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')

#### 👉 Observar:

customer_id → identificador do pedido (chave para orders)  
customer_unique_id → cliente único real  
customer_zip_code_prefix → CEP (prefixo)  
customer_city → cidade  
customer_state → estado 

🔥 Insight

👉 Região impacta diretamente:

- otimizar estoque por região
- reduzir transporte desnecessário
- diminuir emissão de CO₂ (simulado) 
- eficiencia na entrega


“Estados com maior volume de pedidos apresentam maior eficiência logística e menor custo médio de frete.”

### 5.5 sellers

In [21]:
sellers = pd.read_sql("SELECT * FROM sellers", conn)
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [22]:
sellers.dtypes

seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

In [23]:
sellers.columns

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='str')

In [24]:
sellers.describe()

,seller_zip_code_prefix
count,3095.000000
mean,32291.059451
std,32713.453830
min,1001.000000
25%,7093.500000
50%,14940.000000
75%,64552.500000
max,99730.000000


#### 👉 Observar:

seller_id → vendedor (pode ser tratado como fornecedor)
seller_zip_code_prefix → CEP
seller_city → cidade
seller_state → estado 

👉 Permite analisar:

origem dos produtos
distância até cliente (indiretamente)
impacto no frete

## 6. Criação do Dataset para Análise de Consumo Inteligente

### 6.1 Base Analítica

In [25]:
order_items.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

In [26]:
query = """
SELECT 

    oi.order_id,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    
    o.order_purchase_timestamp,
    o.order_status,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
   

    p.product_category_name,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm, 
    
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    
    s.seller_city,
    s.seller_state,
    
    oi.price,
    oi.freight_value
    
FROM order_items oi

JOIN orders o 
    ON oi.order_id = o.order_id

JOIN customers c 
    ON o.customer_id = c.customer_id

JOIN products p 
    ON oi.product_id = p.product_id

JOIN sellers s 
    ON oi.seller_id = s.seller_id

where o.order_status = 'delivered'
"""


In [27]:
df_analytics = pd.read_sql(query, conn)

In [28]:
df_analytics.head()

,order_id,order_item_id,product_id,seller_id,order_purchase_timestamp,order_status,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,...,product_height_cm,product_width_cm,customer_id,customer_unique_id,customer_city,customer_state,seller_city,seller_state,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-02 10:56:33,delivered,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,...,8.0,13.0,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,maua,SP,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-24 20:41:37,delivered,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,...,13.0,19.0,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,belo horizonte,SP,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-08 08:38:49,delivered,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,...,19.0,21.0,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,guariba,SP,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-18 19:28:06,delivered,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,...,10.0,20.0,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,belo horizonte,MG,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-13 21:18:39,delivered,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,...,15.0,15.0,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,mogi das cruzes,SP,19.90,8.72


In [29]:
df_analytics.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'order_purchase_timestamp', 'order_status', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'product_category_name',
       'product_weight_g', 'product_length_cm', 'product_height_cm',
       'product_width_cm', 'customer_id', 'customer_unique_id',
       'customer_city', 'customer_state', 'seller_city', 'seller_state',
       'price', 'freight_value'],
      dtype='str')

In [30]:
df_analytics.dtypes

order_id                             str
order_item_id                      int64
product_id                           str
seller_id                            str
order_purchase_timestamp             str
order_status                         str
order_approved_at                    str
order_delivered_carrier_date         str
order_delivered_customer_date        str
order_estimated_delivery_date        str
product_category_name                str
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
customer_id                          str
customer_unique_id                   str
customer_city                        str
customer_state                       str
seller_city                          str
seller_state                         str
price                            float64
freight_value                    float64
dtype: object

In [31]:
df_analytics.info()

<class 'pandas.DataFrame'>
RangeIndex: 110197 entries, 0 to 110196
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       110197 non-null  str    
 1   order_item_id                  110197 non-null  int64  
 2   product_id                     110197 non-null  str    
 3   seller_id                      110197 non-null  str    
 4   order_purchase_timestamp       110197 non-null  str    
 5   order_status                   110197 non-null  str    
 6   order_approved_at              110182 non-null  str    
 7   order_delivered_carrier_date   110195 non-null  str    
 8   order_delivered_customer_date  110189 non-null  str    
 9   order_estimated_delivery_date  110197 non-null  str    
 10  product_category_name          108660 non-null  str    
 11  product_weight_g               110179 non-null  float64
 12  product_length_cm              110179 non

#### Salvar o dataset Analytics

In [32]:
df_analytics.to_parquet('../data/processed/df_analytics.parquet')

In [33]:
df_analytics.shape

(110197, 23)